In [4]:
import os
import datetime
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('future.no_silent_downcasting', True)

import numpy as np
from laparoscopy_helpers.data_cleaning import (
    to_snake_case, clean_surgical_df
)
from laparoscopy_helpers.plots import *

In [21]:
os.listdir("../Nkhoma_data/poor_patients_funds_data/Endoscopy Ledger")

['Endoscopy Ledger_2025 until September 25.xlsx',
 'Endoscopy Ledger_ 2022-2023.xlsx',
 'Endoscopy Ledger_Consolidated 2024.xlsx',
 '1 Worksheet Endoscopy ledger summary.xlsx']

In [23]:
cleaned_theatre_book = pd.read_excel("../Nkhoma_data/theatre_book_data/combined_clean.xlsx", engine="openpyxl")

In [25]:
endo_path = "../Nkhoma_data/poor_patients_funds_data/Endoscopy Ledger"
endo_files = os.listdir(endo_path)
endos = []
for endo_file in endo_files:
    endo = pd.read_excel(f"{endo_path}/{endo_files[0]}", engine="openpyxl")

In [29]:
endo = pd.read_excel(f"{endo_path}/{endo_files[0]}", engine="openpyxl")

In [30]:
endo

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,NaN,NaN,ENDOSCOPY PATIENTS FOR JANUARY 2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,#,PATIENT NUMBER,NAME,GENDER,DIAGNOSIS,SURGERY,ADMISSION DATE,DISCHARGE DATE,TOTAL BILL (MK),CO-PAY (MK),ENDOSCOPY LEDGER BALANCE (MK),INVOICE NUMBER
3,1,397352,Eliza Evan,Female,Chronic Constipation,Colonoscopy,2025-01-03 00:00:00,2025-01-11 00:00:00,197350,0,197350,398437
4,2,398191,Alinesi Fekinale,Female,Oesophageal Candida,EGD,2025-01-09 00:00:00,2025-01-11 00:00:00,144440,20000,124440,398435
5,3,398805,Annie Willy,Female,Upper and Lower GI Bleed,EGD,2025-01-16 00:00:00,2025-01-17 00:00:00,167700,10000,157700,398509
6,4,398806,Bornface Alick,Male,Dyspepsia,EGD,2025-01-16 00:00:00,2025-01-17 00:00:00,140000,15000,125000,398806
7,5,398697,Wiseman Kambalikana,Male,Anaemia 2 deg UGIB; Acute liver failure,EGD,2025-01-10 00:00:00,2025-01-12 00:00:00,249500,100000,149500,398470
8,6,359515,Wilson Mologeni,Male,Upper GI Bleed,EGD + Banding,2025-01-08 00:00:00,2025-01-10 00:00:00,257100,150000,107100,398398
9,7,387496,Alex Chambaro,Male,UGIB and Previous anemia,Colonoscopy,2025-01-08 00:00:00,2025-01-10 00:00:00,161600,130000,31600,398375


In [34]:
import pandas as pd
import re
from pathlib import Path


def clean_endo_file(filepath: str | Path) -> pd.DataFrame:
    """
    Reads a raw endo Excel/CSV file, finds the header row containing
    '#', 'PATIENT NUMBER', 'NAME', etc., renames columns to snake_case,
    drops rows above the header, and drops rows without an id (#).

    Works for .xlsx, .xls, and .csv files.
    """
    path = Path(filepath)

    # ── 1. Load raw file ──────────────────────────────────────────────
    if path.suffix in (".xlsx", ".xls"):
        raw = pd.read_excel(path, header=None, engine="openpyxl")
    elif path.suffix == ".csv":
        raw = pd.read_csv(path, header=None)
    else:
        # No extension or unknown — try openpyxl first, then CSV
        try:
            raw = pd.read_excel(path, header=None, engine="openpyxl")
        except Exception:
            try:
                raw = pd.read_csv(path, header=None)
            except Exception:
                raise ValueError(
                    f"Could not read '{path.name}' as Excel or CSV. "
                    "Check the file format."
                )

    # ── 2. Find the header row ────────────────────────────────────────
    # The header row is the first row whose first non-null value is '#'
    header_row_idx = None
    for i, row in raw.iterrows():
        values = row.dropna().tolist()
        if values and str(values[0]).strip() == "#":
            header_row_idx = i
            break

    if header_row_idx is None:
        raise ValueError("Could not find the header row (row starting with '#').")

    # ── 3. Rebuild DataFrame from the header row onward ───────────────
    headers = raw.iloc[header_row_idx].tolist()
    data = raw.iloc[header_row_idx + 1 :].copy()
    data.columns = headers
    data = data.reset_index(drop=True)

    # ── 4. Rename columns to snake_case ───────────────────────────────
    def to_snake_case(name: str) -> str:
        name = str(name).strip()
        name = re.sub(r"[^\w\s]", "", name)          # remove punctuation
        name = re.sub(r"\s+", "_", name)              # spaces → underscore
        name = name.lower()
        return name

    data.columns = [to_snake_case(c) for c in data.columns]

    # The '#' column becomes 'id' for clarity
    data = data.rename(columns={"#": "id"})

    # ── 5. Drop rows without a valid id ───────────────────────────────
    data = data[pd.to_numeric(data["id"], errors="coerce").notna()]
    data["id"] = data["id"].astype(int)
    data = data.reset_index(drop=True)

    return data


# ── Batch-process all endo files in a folder ─────────────────────────
def process_all_endo_files(folder: str | Path) -> dict[str, pd.DataFrame]:
    """
    Finds all Excel/CSV files in `folder` whose name contains 'endo'
    (case-insensitive), cleans each one, and returns a dict of
    {filename: cleaned_dataframe}.
    """
    folder = Path(folder)
    results = {}

    for f in sorted(folder.iterdir()):
        if "endo" in f.name.lower() and f.suffix in (".xlsx", ".xls", ".csv"):
            print(f"Processing: {f.name}")
            try:
                df = clean_endo_file(f)
                results[f.name] = df
                print(f"  → {len(df)} rows, columns: {list(df.columns)}")
            except Exception as e:
                print(f"  ✗ Failed: {e}")

    return results



In [35]:


endo_path = "../Nkhoma_data/poor_patients_funds_data/Endoscopy Ledger"
endo_files = os.listdir(endo_path)

endos = []
for endo_file in endo_files:
    df = clean_endo_file(f"{endo_path}/{endo_file}")
    endos.append(df)

combined = pd.concat(endos, ignore_index=True)
combined.head()

KeyError: 'id'